# Student Performance

A short, personalized walkthrough built from this dataset's actual values.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.titleweight': 'bold',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but those won't be saved outside of the current session


## 1. Load and Inspect

Read the data and check shape, sample rows, and missingness.

In [ ]:
from pathlib import Path

csv_name = 'student_performance.csv'
candidate_paths = list(Path('/kaggle/input').rglob(csv_name))
path = str(candidate_paths[0]) if candidate_paths else csv_name

df = pd.read_csv(path)
print('Loaded from:', path)
print('shape:', df.shape)
display(df.head())

profile = pd.DataFrame({
    'missing': df.isna().sum(),
    'dtype': df.dtypes.astype(str),
})
display(profile)


## 2. Dataset Snapshot

These quick notes are derived from the current dataset output:

- The dataset has **500 rows** and **11 columns**.
- Top `student_id` by `final_score` is **STU0376** (95).
- `final_score` median is **56** with a maximum of **95**.
- Missing values exist (total **117**), mainly in `parent_education` (117).

## 3. Visual Summary

A compact visual pass on leaders and metric distribution.

In [ ]:
def to_numeric_human(series):
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors='coerce')
    cleaned = series.astype(str).str.strip().str.replace(',', '', regex=False)
    mult = cleaned.str.extract(r'(?i)([KMB])$')[0].str.upper().map({'K':1e3, 'M':1e6, 'B':1e9}).fillna(1)
    nums = pd.to_numeric(cleaned.str.replace(r'(?i)[KMB]$', '', regex=True), errors='coerce')
    return nums * mult

entity_candidates = ['channel','Channel Name','account','username','page','name','title','repo-name','student_id']
metric_candidates = ['subscribers','followers','likes','views','video_views','stars','final_score','forks','talking_about','uploads','videos','open-issues']

entity_col = next((c for c in entity_candidates if c in df.columns), None)
metric_col = next((c for c in metric_candidates if c in df.columns), None)

if metric_col is None:
    numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and c.lower() != 'rank']
    metric_col = numeric_cols[0] if numeric_cols else None

metric_num = to_numeric_human(df[metric_col]) if metric_col else pd.Series(dtype=float)

secondary_col = None
for c in df.columns:
    if c == metric_col or c.lower() == 'rank':
        continue
    candidate = to_numeric_human(df[c])
    if candidate.notna().sum() >= max(5, int(0.5 * len(df))):
        secondary_col = c
        secondary_num = candidate
        break

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

if entity_col and metric_col and metric_num.notna().any():
    top_plot = pd.DataFrame({entity_col: df[entity_col], '__metric_num': metric_num}).dropna().sort_values('__metric_num', ascending=False).head(10).sort_values('__metric_num')
    sns.barplot(data=top_plot, y=entity_col, x='__metric_num', hue=entity_col, dodge=False, legend=False, ax=axes[0], palette='crest')
    axes[0].set_title(f'Top 10 by {metric_col}')
    axes[0].set_xlabel(f'{metric_col} (numeric)')
    axes[0].set_ylabel('')
else:
    axes[0].text(0.5, 0.5, 'No suitable entity/metric pair found', ha='center', va='center')
    axes[0].set_axis_off()

if metric_col and metric_num.notna().any():
    sns.histplot(metric_num.dropna(), bins=20, kde=True, ax=axes[1], color='#2a9d8f')
    axes[1].set_title(f'Distribution of {metric_col}')
    axes[1].set_xlabel(f'{metric_col} (numeric)')
else:
    axes[1].text(0.5, 0.5, 'No numeric metric found', ha='center', va='center')
    axes[1].set_axis_off()

plt.tight_layout()
plt.show()

if metric_col and secondary_col and metric_num.notna().any():
    secondary_num = to_numeric_human(df[secondary_col])
    scatter_df = pd.DataFrame({'x': secondary_num, 'y': metric_num}).dropna()
    if len(scatter_df) >= 10:
        plt.figure(figsize=(7, 5))
        sns.scatterplot(data=scatter_df, x='x', y='y', s=70, alpha=0.75, color='#264653')
        plt.title(f'{metric_col} vs {secondary_col}')
        plt.xlabel(f'{secondary_col} (numeric)')
        plt.ylabel(f'{metric_col} (numeric)')
        plt.tight_layout()
        plt.show()


## 4. ML Baseline (Predict `passed`)

This part is intentionally split into small steps: class balance, preprocessing, model training, and evaluation.

In [ ]:
# Quick target check
target_counts = df['passed'].value_counts()
display(target_counts.to_frame('count'))
display((target_counts / len(df) * 100).round(1).to_frame('percent'))


### 4.1 Setup Features and Pipeline

Numeric and categorical columns are handled separately, with imputation for missing values.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

X = df.drop(columns=['student_id', 'passed'])
y = df['passed']

num_cols = X.select_dtypes(include=['number']).columns.tolist()
cat_cols = X.select_dtypes(exclude=['number']).columns.tolist()
print('Numeric columns:', num_cols)
print('Categorical columns:', cat_cols)

preprocess = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols),
])


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)


### 4.2 Train and Compare Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=400, random_state=42, class_weight='balanced'),
}

fitted_models = {}
rows = []

for name, model in models.items():
    clf = Pipeline([('preprocess', preprocess), ('model', model)])
    clf.fit(X_train, y_train)

    pred = clf.predict(X_test)
    prob = clf.predict_proba(X_test)[:, 1]

    fitted_models[name] = clf
    rows.append({
        'model': name,
        'accuracy': accuracy_score(y_test, pred),
        'roc_auc': roc_auc_score((y_test == 'Yes').astype(int), prob),
    })

results_df = pd.DataFrame(rows).sort_values(['roc_auc', 'accuracy'], ascending=False).reset_index(drop=True)
display(results_df)


### 4.3 Inspect Best Model

In [ ]:
best_name = results_df.loc[0, 'model']
best_model = fitted_models[best_name]
best_pred = best_model.predict(X_test)

print('Best model:', best_name)
print(classification_report(y_test, best_pred, digits=3))

cm = confusion_matrix(y_test, best_pred, labels=['No', 'Yes'])
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', cbar=False,
    xticklabels=['Pred No', 'Pred Yes'],
    yticklabels=['Actual No', 'Actual Yes']
)
plt.title(f'Confusion Matrix ({best_name})')
plt.tight_layout()
plt.show()


## 5. Key Takeaways

- Class balance is roughly 65% `Yes` vs 35% `No`, so we track ROC-AUC in addition to accuracy.
- On this split (`random_state=42`), Logistic Regression scored about **0.95 accuracy / 0.993 ROC-AUC**.
- Random Forest scored **1.00 accuracy / 1.00 ROC-AUC** on the holdout set; treat that as strong but potentially optimistic until cross-validation confirms it.